In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df = pd.read_csv('resumes_processed.csv')
df = df.dropna(subset=['cleaned_resume'])
print("Data loaded:", df.shape)

Data loaded: (2483, 2)


In [2]:
print("Loading model... (first time downloads ~90MB, be patient)")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model ready!")

Loading model... (first time downloads ~90MB, be patient)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Himela\Documents\hireready\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Himela\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model ready!


In [3]:
sample = "Python developer with machine learning experience"
embedding = model.encode(sample)

print("Type:", type(embedding))
print("Shape:", embedding.shape)
print("First 10 numbers:", embedding[:10])

Type: <class 'numpy.ndarray'>
Shape: (384,)
First 10 numbers: [-0.04796336 -0.02896123  0.03865119  0.06347781 -0.00402706 -0.11416177
 -0.01606943 -0.00708558 -0.09857593 -0.04132647]


In [4]:
sentences = [
    "Python developer with machine learning experience",
    "ML engineer skilled in Python and data science",
    "Chef with experience in Italian cuisine",
    "Accountant with tax filing and Excel skills"
]

embeddings = model.encode(sentences)

# Compare first sentence against all others
base = embeddings[0].reshape(1, -1)

print(f"Base: '{sentences[0]}'\n")
for i in range(1, len(sentences)):
    score = cosine_similarity(base, embeddings[i].reshape(1, -1))[0][0]
    print(f"vs '{sentences[i]}'")
    print(f"   Similarity: {round(score * 100, 2)}%\n")

Base: 'Python developer with machine learning experience'

vs 'ML engineer skilled in Python and data science'
   Similarity: 73.58000183105469%

vs 'Chef with experience in Italian cuisine'
   Similarity: 25.420000076293945%

vs 'Accountant with tax filing and Excel skills'
   Similarity: 26.860000610351562%



In [5]:
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

def match_resume_jd(resume_text, jd_text):
    # Clean both
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    # Generate embeddings
    resume_embedding = model.encode(cleaned_resume).reshape(1, -1)
    jd_embedding = model.encode(cleaned_jd).reshape(1, -1)
    
    # Compute similarity
    score = cosine_similarity(resume_embedding, jd_embedding)[0][0]
    return round(float(score * 100), 2)

In [6]:
sample_jd = """
Looking for a Python Developer with experience in machine learning,
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""

print("Embedding matcher results:\n")

categories = ['INFORMATION-TECHNOLOGY', 'ENGINEERING', 'HR', 'ACCOUNTANT', 'CHEF']

for category in categories:
    cat_resumes = df[df['category'] == category]
    if len(cat_resumes) > 0:
        resume = cat_resumes.iloc[0]['cleaned_resume']
        score = match_resume_jd(resume, sample_jd)
        print(f"{category:25} → {score}%")

Embedding matcher results:

INFORMATION-TECHNOLOGY    → 26.22%
ENGINEERING               → 20.56%
HR                        → 17.0%
ACCOUNTANT                → 3.77%
CHEF                      → 22.79%


In [7]:
import pickle

# Save model name so we can reload it
with open('embedding_model_name.txt', 'w') as f:
    f.write('all-MiniLM-L6-v2')

print("Saved!")

Saved!
